# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
One row = one client-content-day of performance: report_date × client_hash_id ×
content_hash_id, from fact_content_daily_performance, restricted to month=2026-03 — a
mid-panel month, not the sealed final-month _sample (June 2026 is reserved as the outcome
window for any past→future label, per the panel warning). I verify this grain claim, its row
count/date span, its missingness pattern, and its availability filter in Section 3 below.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=march_files[0],
    token=os.environ["HF_TOKEN"],
)
df_month = pd.read_parquet(path)
print(df_month.shape)
print(df_month["report_date"].min(), df_month["report_date"].max())

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (all same-day-knowable):
- gsc_clicks, gsc_impressions — used to compute CTR (gsc_clicks / gsc_impressions); no ctr
  column exists directly in this table.
- gsc_avg_position — Search Console reports it same-day.
- ga4_engaged_sessions, ga4_total_engagement_sec — engagement signal, in place of a scroll_rate
  column that doesn't exist here.
- sessions_ai — raw AI-referral session count; would need dividing by total sessions across
  all sessions_* columns to get a percentage, since no ai_traffic_pct column exists.

Label / proxy:
- No trained label yet. No is_declining_label column exists in this table either — that field
  lives only in the starter CSV. Any label here has to be built from an observed future window,
  not inherited from the starter dataset's rule-derived label.

Context (not a feature, not a label, but needed to interpret rows):
- client_hash_id, content_hash_id — for grouping, joining, and per-client filtering only; never
  features (pseudonyms).
- report_date — needed to define the window, not a predictive signal itself.
- ga4_data_available — the real row-level availability flag; needed to filter rows correctly
  (see Section 3).
- client_has_ga4 — a client-level flag that turned out to disagree with ga4_data_available for
  roughly half of the clients where it reads False (see Section 3's last query) — kept as
  context only, not trusted as a filter.

Excluded:
- GA4 columns (ga4_pageviews, ga4_sessions, ga4_engaged_sessions, etc.) on rows where
  ga4_data_available is not True — these are zero-filled placeholders or missing, not real
  "zero engagement" (confirmed in Section 3: 95.8% of March rows fail this filter).
- fact_content_daily_performance_sample — the final month, the natural outcome window for any
  label I'll eventually build; using it now means developing inside my own future test window.
- fact_content_query_90d — excluded from this month's slice entirely for now, per its own
  window-overlap and repeated-context-column traps.

Output: a per-content-day row of feature values (CTR computed from clicks/impressions,
gsc_avg_position, engagement signals, AI-session share) for March 2026, filtered to rows with
real GA4 availability — handed off as the feature frame for opportunity scoring, not yet a
trained prediction.

In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=os.environ["HF_TOKEN"])
march_files = [f for f in files if "2026-03" in f and "fact_content_daily_performance" in f and "sample" not in f]
print(march_files)

['fact_content_daily_performance/month=2026-03/data_0.parquet']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
Verify grain: one row really is one report_date  client_id  content_id.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
import os

token = os.environ["HF_TOKEN"]  # from Colab Secret, never pasted literal

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data.parquet",
    token=token,
)
df_month = pd.read_parquet(path)

grain_cols = ["report_date", "client_id", "content_id"]
dupe_counts = df_month.groupby(grain_cols).size()
print("Groups with >1 row (should be 0 if grain claim holds):", (dupe_counts > 1).sum())

In [5]:
grain_cols = ["report_date", "client_hash_id", "content_hash_id"]
dupe_counts = df_month.groupby(grain_cols).size()
print("Groups with >1 row (should be 0):", (dupe_counts > 1).sum())

Groups with >1 row (should be 0): 0


Verify missingness — is it random, or does it follow a pattern (e.g. by client)?

In [ ]:
missing_by_client = df_month.groupby("client_id")["gsc_avg_position"].apply(
    lambda s: (s == 0).mean()  # 0 is a "no data" sentinel, not a real rank
)
print("Share of avg_position==0 rows, top 5 clients by missingness:")
print(missing_by_client.sort_values(ascending=False).head())

In [6]:
missing_by_client = df_month.groupby("client_hash_id")["gsc_avg_position"].apply(
    lambda s: (s == 0).mean()
)
print("Share of gsc_avg_position==0 rows, top 5 clients by missingness:")
print(missing_by_client.sort_values(ascending=False).head())

Share of gsc_avg_position==0 rows, top 5 clients by missingness:
client_hash_id
client_73cda7b4e4f265ea    0.047990
client_62f4a7e64f5e0096    0.047509
client_a80fca3f171ed1de    0.045778
client_fef1a8f436438636    0.044150
client_1a730cb2640a1abf    0.039079
Name: gsc_avg_position, dtype: float64


Verify availability — filter GA4 rows with IS TRUE, show survivors.

In [7]:
print(df_month["ga4_data_available"].value_counts())
print(df_month["client_has_ga4"].value_counts())

cross = df_month.groupby("client_hash_id").agg(
    has_ga4=("client_has_ga4", "first"),
    pct_rows_available=("ga4_data_available", "mean")
)
print(cross.head(10))

ga4_data_available
False    6408671
True      413966
Name: count, dtype: int64
client_has_ga4
True     6822637
False    3018741
Name: count, dtype: int64
                         has_ga4 pct_rows_available
client_hash_id                                     
client_0797ff3a1fc9a6a5    False                NaN
client_08a6a72ff48e62c0    False                NaN
client_08d2847f24cf89c1     True            0.02959
client_0e1acc6cd57b0eba     True           0.028528
client_0fa64a184f18a4a0     True           0.070163
client_157ffe4d4a595515    False           0.058267
client_19b89ee4fe3db6da     True           0.000014
client_1a730cb2640a1abf    False           0.113682
client_20259bd6705d81d4     True           0.218337
client_2094c6eb080311d5     True           0.054842


In [8]:
print(df_month["ga4_data_available"].isna().sum(), "rows with NaN (not True or False) in ga4_data_available")

available = df_month[df_month["ga4_data_available"] == True]
print(f"Rows before filter: {len(df_month)}")
print(f"Rows after ga4_data_available IS TRUE: {len(available)}")
print(f"Dropped: {len(df_month) - len(available)} ({(1 - len(available)/len(df_month)):.1%})")

3018741 rows with NaN (not True or False) in ga4_data_available
Rows before filter: 9841378
Rows after ga4_data_available IS TRUE: 413966
Dropped: 9427412 (95.8%)


In [9]:
# does every client_has_ga4=False client show 100% NaN?
check = df_month.groupby("client_hash_id").agg(
    has_ga4=("client_has_ga4", "first"),
    pct_nan=("ga4_data_available", lambda s: s.isna().mean())
)
print(check[check["has_ga4"] == False]["pct_nan"].describe())

count    20.000000
mean      0.693803
std       0.401671
min       0.052881
25%       0.234037
50%       1.000000
75%       1.000000
max       1.000000
Name: pct_nan, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me why a page's traffic changed — only that it did. It can't support
causal claims like "refreshing this page caused recovery."

Availability is far more restrictive than expected: only 4.2% of March 2026 rows
(413,966 / 9,841,378) have ga4_data_available == True. The rest split between rows with no
GA4 relationship at all (NaN, 3,018,741 rows) and rows where GA4 exists for the client but
this particular row predates their data start (False, 6,408,671 rows). Any GA4-based feature
in this analysis only has real support on a small fraction of the month.

The dataset also contains an unresolved inconsistency between its two GA4 flags. Of the 20
clients flagged client_has_ga4 = False, only about half show ga4_data_available as NaN for
100% of their rows as the flag's name implies — the rest have anywhere from 5% to 100% of
rows carrying a real True/False value despite the client-level flag saying no GA4 relationship
exists. I can't resolve which flag is "correct," so I filter strictly on the row-level
ga4_data_available signal and don't treat client_has_ga4 as reliable context beyond a rough
sanity check.

Missingness in gsc_avg_position (the sentinel-zero case) is real but modest and fairly evenly
distributed across clients (4.0-4.8% among the highest-missingness clients) — not concentrated
in a small number of problem clients.

History depth is unbalanced across clients more broadly (per the flyrank-data skill, a third
of clients have little or no usable search/analytics history) — comparing raw trends across
clients without checking each client's data start first would still be misleading, even though
this month's slice is internally well-bounded.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.